# Kalman Filters for Dynamic Hedge Ratios: Tracking a Beta That Refuses to Sit Still

Every pairs trade hides a regression inside it: the hedge ratio that turns two prices into one spread. Estimate that ratio once and you have quietly assumed the relationship never moves — over fifteen years of EWA and EWC it moves from 0.71 to 1.49 . We build a Kalman filter from scratch in NumPy that treats the hedge ratio as a moving target, then run an honest experiment: identical trading rules on the Kalman spread and the static spread, with costs, and let the data pick the winner.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import adfuller

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Why static hedge ratios die

EWA (Australia) and EWC (Canada) are the canonical cointegration pair — two commodity-heavy developed markets whose daily returns correlate at `0.83`. Regress EWC on EWA over the full sample and you get one hedge ratio for fifteen years: $\beta = 1.57$ , with an ADF p-value of `0.038` on the residual spread — cointegrated, by the book (Engle–Granger: regress one price on the other, then unit-root-test the residual). The problem is the word **one**. A hedge ratio is not a constant of nature; it is a snapshot of an economic relationship — commodity mix, currency betas, index composition — and every one of those drifted between 2010 and 2024. The standard fix is a rolling window, and it limps for a reason worth understanding: every observation inside the window carries equal weight, so year-old data moves today's estimate exactly as much as yesterday's does — and the day an old observation falls out of the window, the estimate jerks for no economic reason at all. That artefact has a name: the **window cliff**.

In [ ]:
px = yf.download(["EWA", "EWC"], start="2010-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].dropna()
ewa, ewc = px["EWA"].to_numpy(), px["EWC"].to_numpy()

(px / px.iloc[0]).plot(title="EWA & EWC, normalized");

## 2. Beta as a state, not a constant

The Kalman filter starts from a different premise, and it is worth sitting with for a moment: the “true” hedge ratio is something you can never observe directly — you only see prices, which are noisy evidence about it. So treat the regression coefficients as **unobserved states**drifting through time, and treat each day's prices as one more noisy measurement of where they are. (A GPS does exactly this with your position; here the hidden position is β.) Two equations define the model. The **state equation** says the hedge ratio and intercept follow a random walk — $[\beta_t, \alpha_t] = [\beta_{t-1}, \alpha_{t-1}] + \omega_t$ ​ , α t ​ ] = [ β t − 1 ​ , α t − 1 ​ ] + ω t ​ — tomorrow's relationship is today's, plus noise. The **observation equation** says $\mathrm{EWC}_t = \beta_t\,\mathrm{EWA}_t + \alpha_t + \varepsilon_t$ t ​ = β t ​ EWA t ​ + α t ​ + ε t ​ . Two variances close the model: the state noise $Q = \frac{\delta}{1-\delta}\,I$ δ ​ I with $\delta = 10^{-5}$ (the standard parameterization from Chan), and observation noise $R = 10^{-3}$ .

Delta is the single real knob, and it replaces the window size entirely: it is a **forgetting rate**. Larger δ lets the states wander faster (adaptive but noisy); smaller δ pins them down (smooth but laggy); δ = 0 collapses to recursive least squares — a static beta refined forever. Both values here are textbook defaults, deliberately not tuned on this sample: tune δ to the backtest and you are optimizing the strategy through the back door.

In [ ]:
x_const = sm.add_constant(px["EWA"])

ols = sm.OLS(px["EWC"], x_const).fit()
beta_static, alpha_static = float(ols.params["EWA"]), float(ols.params["const"])

spread_static = px["EWC"] - beta_static * px["EWA"] - alpha_static
print(f"static beta  = {beta_static:.4f}   alpha = {alpha_static:.4f}")
print(f"ADF p-value on the static spread = {adfuller(spread_static.to_numpy())[1]:.4f}")

roll = RollingOLS(px["EWC"], x_const, window=252).fit(params_only=True)
beta_roll = roll.params["EWA"].to_numpy()
alpha_roll = roll.params["const"].to_numpy()

## 3. The filter in five lines of NumPy

No library, no black box — the whole filter is a predict step and an update step, looped over the sample. Predict: with a random-walk transition the state estimate is unchanged and its covariance grows by `Q` — a day passes, so the filter becomes a little less sure of where β is. Update: compare the observed EWC to the prediction, and shift the states toward the error in proportion to the **Kalman gain** — the ratio of state uncertainty to total uncertainty. Big surprise while unsure of yourself: move a lot. Big surprise while confident: blame measurement noise, barely move.

The gain is the elegance. When the filter is uncertain (large `p_cov`), it learns aggressively from each observation; once confident, new data barely moves it — unless `Q` keeps injecting doubt, which is exactly what lets β keep adapting forever. An exponentially-weighted regression, derived from first principles rather than picked from a menu of window sizes.

In [ ]:
def kalman_hedge(x, y, delta=1e-05, r_obs=0.001):
    n = len(x)
    q = (delta / (1.0 - delta)) * np.eye(2)   # trans_cov
    state = np.zeros(2)                       # [beta, alpha], diffuse start
    p_cov = np.eye(2)
    betas, alphas = np.zeros(n), np.zeros(n)
    for t in range(n):
        h = np.array([x[t], 1.0])             # observation map
        p_cov = p_cov + q                     # predict (F = I)
        e = y[t] - h @ state                  # innovation
        s = h @ p_cov @ h + r_obs             # innovation variance
        k = p_cov @ h / s                     # Kalman gain
        state = state + k * e                 # update
        p_cov = p_cov - np.outer(k, h @ p_cov)
        betas[t], alphas[t] = state
    return betas, alphas

beta_kf, alpha_kf = kalman_hedge(ewa, ewc)
print(f"Kalman beta: first tradeable {beta_kf[252]:.3f} ... last {beta_kf[-1]:.3f}")

## 4. Three betas, one pair

Slice off the first year (rolling-OLS warm-up, Kalman burn-in) and compare the three estimators of the same quantity. The static line says the answer is 1.57 , forever. The rolling OLS swings between `0.19` and `2.75` — whipping around every regime change a full window late. The Kalman path stays in a far narrower band, moving early and smoothly: no cliff, because no window.

Note what the rolling estimator does around 2020–2021: the COVID shock enters the window, distorts the regression for exactly 252 trading days, then falls out and the estimate jumps again — two artefacts from one event. The filter digests the same shock in weeks and moves on.

In [ ]:
t0 = 252
dates = px.index[t0:]
plt.plot(dates, beta_kf[t0:], label="Kalman", lw=1.6)
plt.plot(dates, beta_roll[t0:], label="rolling OLS (252d)", lw=1.2)
plt.axhline(beta_static, color="k", ls="--", lw=1, label="static OLS")
plt.legend(); plt.title("EWC~EWA hedge ratio, three estimators");
plt.show()

## 5. Trading the spread

Now make the estimator earn its living — a better β is only worth money if it makes a better spread. The spread is $\mathrm{EWC}_t - \beta_t\,\mathrm{EWA}_t - \alpha_t$ t ​ − β t ​ EWA t ​ − α t ​ , z-scored on a trailing 60 -day window. Rules, identical for both variants: enter long the spread (long EWC, short β·EWA) when $z , short when $z > +2$ , exit when z crosses zero. Positions are sized to $1 gross notional at entry, the Kalman variant re-hedges the EWA leg to the current β daily, and every unit of traded notional pays 10 bp. Signals use the close and P&L starts the next day — no lookahead in the rule. The static beta itself, of course, is one giant lookahead: it was fit on all fifteen years, including the future of every trade it takes.

In [ ]:
def backtest(beta, alpha, dynamic, z_win=60, entry=2,
             cost=0.001, start=252):
    n = len(ewa)
    spread = pd.Series(ewc - beta * ewa - alpha)
    z = ((spread - spread.rolling(z_win).mean())
         / spread.rolling(z_win).std(ddof=1)).to_numpy()

    pos, p = np.zeros(n), 0.0
    for t in range(start, n):
        zt = z[t]
        if np.isnan(zt):
            pos[t] = p; continue
        if p == 0.0:
            if zt < -entry: p = 1.0
            elif zt > entry: p = -1.0
        elif p == 1.0 and zt >= 0.0: p = 0.0
        elif p == -1.0 and zt <= 0.0: p = 0.0
        pos[t] = p

    ret = np.zeros(n)
    n_ewc = n_ewa = g0 = 0.0
    trades = 0
    for t in range(start, n):
        pnl = n_ewc * (ewc[t] - ewc[t-1]) + n_ewa * (ewa[t] - ewa[t-1])
        c = 0.0
        if pos[t] != pos[t-1]:
            if pos[t-1] == 0.0:                     # entry
                g0 = ewc[t] + abs(beta[t]) * ewa[t]
                tc, ta = pos[t] / g0, -pos[t] * beta[t] / g0
                trades += 1
            else:                                   # exit
                tc = ta = 0.0
            c = cost * (abs(tc - n_ewc) * ewc[t] + abs(ta - n_ewa) * ewa[t])
            n_ewc, n_ewa = tc, ta
        elif dynamic and pos[t] != 0.0:             # re-hedge to beta_t
            ta = -pos[t] * beta[t] / g0
            c = cost * abs(ta - n_ewa) * ewa[t]
            n_ewa = ta
        ret[t] = pnl - c

    rr = ret[start:]
    eq = np.cumprod(1.0 + rr)
    ann, vol = rr.mean() * 252, rr.std(ddof=1) * np.sqrt(252)
    mdd = (eq / np.maximum.accumulate(eq) - 1.0).min()
    return eq, dict(ann_ret=ann, ann_vol=vol, sharpe=ann / vol,
                    max_dd=mdd, trades=trades)

eq_kf, st_kf = backtest(beta_kf, alpha_kf, dynamic=True)
eq_st, st_st = backtest(np.full(len(ewa), beta_static),
                        np.full(len(ewa), alpha_static), dynamic=False)

plt.plot(px.index[252:], eq_kf, label="Kalman beta")
plt.plot(px.index[252:], eq_st, label="static beta")
plt.legend(); plt.title("Spread mean reversion, net of 10 bp costs");
plt.show()

## 6. What the numbers actually say

Here is the honest scoreboard, and it is more interesting than a clean win. Gross of costs, the Kalman spread is the better signal on every risk-adjusted axis — more Sharpe, half the volatility, half the drawdown. But the filtered spread mean-reverts **fast**, so it trades 152 round trips to the static variant's 57 — and at 10 bp per unit of traded notional, that turnover consumes the entire edge and then some.

So did the Kalman beta “improve” the strategy? As an estimator, unambiguously — better gross Sharpe, half the drawdown, and no lookahead, against a static baseline that was handed the answer key. As a net P&L line at retail costs, no: the same adaptivity that tracks the relationship also generates signals faster than 10 bp round trips can pay for. Halve the cost and the gap halves; at institutional frictions of 1–2 bp the Kalman variant pulls level and ahead. Estimation quality and implementability are different axes, and a backtest that reports only one is hiding the other.

In [ ]:
eq_kf_g, st_kf_g = backtest(beta_kf, alpha_kf, dynamic=True, cost=0.0)
eq_st_g, st_st_g = backtest(np.full(len(ewa), beta_static),
                            np.full(len(ewa), alpha_static),
                            dynamic=False, cost=0.0)

rows = pd.DataFrame([st_kf, st_st, st_kf_g, st_st_g],
                    index=["Kalman (10bp)", "static (10bp)",
                           "Kalman (gross)", "static (gross)"])
rows.style.format({"ann_ret": "{:.2%}", "ann_vol": "{:.2%}", "sharpe": "{:.2f}",
                   "max_dd": "{:.2%}"})

- Kalman, R. E. (1960). A New Approach to Linear Filtering and Prediction Problems. Journal of Basic Engineering, 82(1).
- Engle, R. F. & Granger, C. W. J. (1987). Co-integration and Error Correction: Representation, Estimation, and Testing. Econometrica, 55(2), 251–276.
- Chan, E. (2013). Algorithmic Trading: Winning Strategies and Their Rationale. Wiley — ch. 3, the EWA/EWC Kalman example and the δ/(1−δ) parameterization.
- Harvey, A. C. (1989). Forecasting, Structural Time Series Models and the Kalman Filter. Cambridge University Press.
- Companion notebook: `kalman-filter-hedge-ratios.ipynb` — reproduces every figure from raw data; fully deterministic, no RNG.